In [170]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import gspread

from datetime import datetime

In [171]:
data = []
page = 1
max_page = 1

In [183]:
while page <= max_page:
    res = requests.get(f'https://ryvok.ru/instrumenty/shlifmashiny/?page={page}')
    soup = BeautifulSoup(res.text, 'html')

    elements = soup.find_all('div', class_='app-product-card app-product-card_with-horizontal')

    for e in elements:
        data.append({
            'titele': e.find('div', class_='app-product-card__title').text.strip(),
            'status': e.find('div', class_='app-product-card__status').text.strip(),
            'price ': e.find('div', class_='app-product-card__price').text.strip().replace('\xa0', '').replace(' ₽', ''),
        })
    
    pagination = soup.find('div', class_ = 'ui-pagination__numbers')
    
    pages = [p.text.strip() for p in pagination.find_all('div', class_ = 'ui-pagination__number')]
    int_pages = []
    
    for p in pages:
        try:
            n = int(p)
            int_pages.append(n)
        except:
            continue

    max_page = max(int_pages)
    page += 1

In [184]:
df = pd.DataFrame(data)
df['timestamp'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

In [185]:
gc = gspread.service_account(filename='cred.json')
wks = gc.open("web_parsing_report").sheet1

current_df = pd.DataFrame(wks.get_all_records())

In [186]:
merged_df = pd.concat([df, current_df])
wks.update([merged_df.columns.values.tolist()] + merged_df.values.tolist())

{'spreadsheetId': '1xTuWDIh3AVlbklLLv3axkU3UGFRfdCCUJMqWmYLmGFg',
 'updatedRange': "'Лист1'!A1:D1721",
 'updatedRows': 1721,
 'updatedColumns': 4,
 'updatedCells': 6884}

In [188]:
merged_df.shape

(1720, 4)